In [0]:
%sql
CREATE OR REPLACE TABLE proyecto_bi.gold.agg_viajes_por_dia AS
SELECT
    DATE(tpep_pickup_datetime)          AS fecha,
    COUNT(*)                            AS total_viajes,
    ROUND(SUM(total_amount), 2)         AS ingreso_total,
    ROUND(AVG(total_amount), 2)         AS ingreso_promedio,
    ROUND(AVG(trip_distance), 2)        AS distancia_promedio,
    ROUND(AVG(passenger_count), 2)      AS pasajeros_promedio,
    ROUND(AVG(DATEDIFF(MINUTE, 
        tpep_pickup_datetime, 
        tpep_dropoff_datetime)), 2)     AS duracion_promedio_min,
    ROUND(SUM(tip_amount), 2)           AS propinas_total
FROM proyecto_bi.silver.yellow_trips
GROUP BY DATE(tpep_pickup_datetime);

In [0]:
%sql
SELECT * FROM proyecto_bi.gold.agg_viajes_por_dia
ORDER BY fecha
LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE proyecto_bi.gold.agg_viajes_por_hora AS
SELECT
    HOUR(tpep_pickup_datetime)          AS hora,
    COUNT(*)                            AS total_viajes,
    ROUND(AVG(total_amount), 2)         AS ingreso_promedio,
    ROUND(AVG(trip_distance), 2)        AS distancia_promedio
FROM proyecto_bi.silver.yellow_trips
GROUP BY HOUR(tpep_pickup_datetime);

In [0]:
%sql
CREATE OR REPLACE TABLE proyecto_bi.gold.agg_viajes_por_pago AS
SELECT
    CASE payment_type
        WHEN 1 THEN 'Tarjeta de crédito'
        WHEN 2 THEN 'Efectivo'
        WHEN 3 THEN 'Sin cargo'
        WHEN 4 THEN 'Disputa'
        WHEN 5 THEN 'Desconocido'
    END                                 AS metodo_pago,
    COUNT(*)                            AS total_viajes,
    ROUND(SUM(total_amount), 2)         AS ingreso_total,
    ROUND(AVG(tip_amount), 2)           AS propina_promedio
FROM proyecto_bi.silver.yellow_trips
GROUP BY payment_type;

In [0]:
%sql
CREATE OR REPLACE TABLE proyecto_bi.gold.agg_viajes_por_zona AS
SELECT
    PULocationID                        AS zona_id,
    COUNT(*)                            AS total_viajes,
    ROUND(SUM(total_amount), 2)         AS ingreso_total,
    ROUND(AVG(trip_distance), 2)        AS distancia_promedio
FROM proyecto_bi.silver.yellow_trips
GROUP BY PULocationID
ORDER BY total_viajes DESC
LIMIT 20;

In [0]:
%sql
SELECT 'agg_por_hora'  AS tabla, COUNT(*) AS filas FROM proyecto_bi.gold.agg_viajes_por_hora  UNION ALL
SELECT 'agg_por_pago'  AS tabla, COUNT(*) AS filas FROM proyecto_bi.gold.agg_viajes_por_pago   UNION ALL
SELECT 'agg_por_zona'  AS tabla, COUNT(*) AS filas FROM proyecto_bi.gold.agg_viajes_por_zona;